In [12]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
from IPython.display import display, HTML

load_dotenv()

def create_db_engine():
    user     = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host     = os.getenv('DB_HOST', 'localhost')
    port     = os.getenv('DB_PORT', '5432')
    dbname   = os.getenv('DB_NAME', 'data_lake')
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str)

engine = create_db_engine()
print("✓ Engine criada com sucesso.")

✓ Engine criada com sucesso.


In [17]:
tabelas = [
    "dfp_cia_aberta_bpa_con"  
]

In [15]:
def processar_e_exibir(engine, lista_tabelas):
    for tabela in lista_tabelas:
        # Título estilizado usando Markdown/HTML para separar as tabelas
        display(HTML(f"<h3 style='color: #2e7d32;'>📊 Tabela: {tabela}</h3>"))
        
        query = text(f"""
            WITH 
                analitico AS (
                    -- Detalha cada caso de reapresentação por data
                    SELECT 
                        "DENOM_CIA", 
                        "CNPJ_CIA", 
                        "DT_REFER", 
                        MAX("VERSAO"::INTEGER) AS max_versao_periodo, 
                        COUNT(DISTINCT "VERSAO") AS qtde_versoes_distintas_periodo
                    FROM 
                        layer_01_bronze.dfp_cia_aberta_bpa_con 
                    GROUP BY 
                        "DENOM_CIA", "DT_REFER", "CNPJ_CIA"
                    HAVING 
                        MAX("VERSAO"::INTEGER) > 1
                ),
                sintetico AS (
                    -- Resume o comportamento histórico da empresa
                    SELECT 
                        "DENOM_CIA", 
                        COUNT(DISTINCT "DT_REFER") AS total_datas_retificadas_historico, 
                        MAX("VERSAO"::INTEGER) AS maior_versao_ja_registrada
                    FROM 
                        layer_01_bronze.dfp_cia_aberta_bpa_con
                    WHERE 
                        ("VERSAO"::INTEGER) > 1 
                    GROUP BY 
                        "DENOM_CIA"
                )
                SELECT 
                    a."DENOM_CIA",
                    a."CNPJ_CIA",
                    a."DT_REFER",
                    a.max_versao_periodo,
                    a.qtde_versoes_distintas_periodo,
                    s.total_datas_retificadas_historico,
                    s.maior_versao_ja_registrada
                FROM 
                    analitico a
                JOIN 
                    sintetico s ON a."DENOM_CIA" = s."DENOM_CIA"
                ORDER BY 
                    s.total_datas_retificadas_historico DESC, 
                    a."DENOM_CIA", 
                    a."DT_REFER" DESC;
        """)
        
        try:
            # Carrega os dados
            df_silver = pd.read_sql(query, engine)
            
            # Exibe metadados antes da tabela
            print(f"Total de registros deduplicados: {len(df_silver)}")
            
            # O 'display' do Pandas no Jupyter cria aquela tabela bonita com scroll e cores
            display(df_silver.head(5)) 
            
            display(HTML("<hr>")) # Linha horizontal para separar as iterações
            
        except Exception as e:
            display(HTML(f"<b style='color: red;'>✗ Erro em {tabela}:</b> {e}"))

In [ ]:
def tabela_deduplicada(engine, lista_tabelas):
    for tabela in lista_tabelas:
        print(f" --- Processando tabela {tabela}")
        query = f"""
           
        """

        try:
            df_silver = pd.read_sql(query, engine)
            print(f"✓ Sucesso! Linhas originais: [Ver Query 1] | Linhas após deduplicação: {len(df_silver)}")
            display(df_silver) 

        except Exception as e:
            print(f"✗ Erro ao processar a tabela {tabela}: {e}")

In [16]:
processar_e_exibir(engine, tabelas)

Total de registros deduplicados: 1641


,DENOM_CIA,CNPJ_CIA,DT_REFER,max_versao_periodo,qtde_versoes_distintas_periodo,total_datas_retificadas_historico,maior_versao_ja_registrada
0,BCO SANTANDER (BRASIL) S.A.,90.400.888/0001-42,2024-12-31,2,1,15,4
1,BCO SANTANDER (BRASIL) S.A.,90.400.888/0001-42,2023-12-31,2,1,15,4
2,BCO SANTANDER (BRASIL) S.A.,90.400.888/0001-42,2022-12-31,2,1,15,4
3,BCO SANTANDER (BRASIL) S.A.,90.400.888/0001-42,2021-12-31,2,1,15,4
4,BCO SANTANDER (BRASIL) S.A.,90.400.888/0001-42,2020-12-31,2,1,15,4
